In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
np.random.seed(42)

n = 500

df = pd.DataFrame({
    "Income": np.random.randint(20000, 150001, n),
    "Credit_Score": np.random.randint(300, 851, n),
    "Employment_Status": np.random.choice(["Employed", "Self-Employed", "Unemployed"], n, p=[0.6, 0.3, 0.1]),
    "Loan_Amount": np.random.randint(5000, 100001, n),
    "Loan_Term": np.random.choice([12, 24, 36, 48, 60], n),
    "Previous_Default": np.random.choice([0, 1], n, p=[0.8, 0.2])
})

risk_score = (
    (df["Credit_Score"] >= 650).astype(int) * 2
    + (df["Income"] >= 50000).astype(int)
    + (df["Loan_Amount"] <= 50000).astype(int)
    + (df["Employment_Status"] != "Unemployed").astype(int)
    - df["Previous_Default"] * 3
)

df["Loan_Status"] = (risk_score >= 3).astype(int)

display(df.head(10))

,Income,Credit_Score,Employment_Status,Loan_Amount,Loan_Term,Previous_Default,Loan_Status
0,141958,460,Self-Employed,44214,12,1,0
1,35795,555,Employed,86871,60,0,0
2,20860,622,Self-Employed,33602,60,0,0
3,123694,427,Self-Employed,58331,12,0,0
4,148106,317,Self-Employed,53901,12,0,0
5,139879,622,Self-Employed,68416,24,0,0
6,130268,587,Self-Employed,37196,36,0,1
7,96820,454,Employed,10626,60,0,1
8,74886,789,Employed,12424,24,0,1
9,26265,685,Self-Employed,17857,36,0,1


In [3]:
print("Dataset shape:", df.shape)
print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nLoan status distribution:")
print(df["Loan_Status"].value_counts())

display(df.describe())

Dataset shape: (500, 7)

Column data types:
Income               int32
Credit_Score         int32
Employment_Status      str
Loan_Amount          int32
Loan_Term            int64
Previous_Default     int64
Loan_Status          int64
dtype: object

Missing values:
Income               0
Credit_Score         0
Employment_Status    0
Loan_Amount          0
Loan_Term            0
Previous_Default     0
Loan_Status          0
dtype: int64

Loan status distribution:
Loan_Status
0    265
1    235
Name: count, dtype: int64


,Income,Credit_Score,Loan_Amount,Loan_Term,Previous_Default,Loan_Status
count,500.000000,500.000000,500.000000,500.000000,500.000000,500.000000
mean,87140.206000,565.626000,52491.002000,37.224000,0.194000,0.470000
std,38701.212724,164.714772,27122.965751,17.121218,0.395825,0.499599
min,20206.000000,300.000000,5281.000000,12.000000,0.000000,0.000000
25%,55883.250000,417.000000,31002.000000,24.000000,0.000000,0.000000
50%,87146.500000,552.500000,51165.000000,36.000000,0.000000,0.000000
75%,121253.250000,707.250000,77181.750000,48.000000,0.000000,1.000000
max,149981.000000,850.000000,99880.000000,60.000000,1.000000,1.000000


In [4]:
X = df.drop("Loan_Status", axis=1)
y = df["Loan_Status"]

X = pd.get_dummies(X, columns=["Employment_Status"], dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 400
Testing records: 100


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled successfully")
print("Testing data scaled successfully")

Training data scaled successfully
Testing data scaled successfully


In [6]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="linear", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(name, "trained successfully")

Decision Tree trained successfully
Naive Bayes trained successfully
SVM trained successfully
KNN trained successfully


In [7]:
results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)

    results.append({
        "Algorithm": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

display(results_df.round(3))

,Algorithm,Accuracy,Precision,Recall,F1 Score
0,Decision Tree,1.00,1.000,1.000,1.000
1,KNN,0.94,0.918,0.957,0.938
2,SVM,0.92,0.953,0.872,0.911
3,Naive Bayes,0.67,0.588,1.000,0.740
